In [1]:
import os
print(os.getcwd())


/Users/kirtikapuniani/Cryptography-Testing


In [3]:
# MasterKey.py
from cryptography.fernet import Fernet
import os

BASE_DIR = os.getcwd()
MASTER_KEY_FILE = os.path.join(BASE_DIR, "master.key")

def get_master_key() -> bytes:
    if not os.path.exists(MASTER_KEY_FILE):
        key = Fernet.generate_key()
        with open(MASTER_KEY_FILE, "wb") as f:
            f.write(key)
    else:
        with open(MASTER_KEY_FILE, "rb") as f:
            key = f.read()

    return key

In [5]:
from MasterKey import get_master_key

get_master_key()

b'zW3HoJK8A7ifTWvhveRtKVbg07et588zkCa2IdvUU2Q='

In [4]:
# KeyStorage

import os
import json
from cryptography.fernet import Fernet
from MasterKey import get_master_key

BASE_DIR = os.getcwd()
MASTER_KEY_FILE = os.path.join(BASE_DIR, "keys.enc.json")
KEY_FILE = MASTER_KEY_FILE

def _get_master_fernet():
    return Fernet(get_master_key())


def load_keys():
    if not os.path.exists(KEY_FILE):
        return []

    with open(KEY_FILE, "rb") as f:
        encrypted_data = f.read()

    decrypted = _get_master_fernet().decrypt(encrypted_data)
    return json.loads(decrypted.decode())


def save_keys(data: list):
    encrypted = _get_master_fernet().encrypt(json.dumps(data).encode())
    with open(KEY_FILE, "wb") as f:
        f.write(encrypted)


def add_key_entry(entry: dict):
    store = load_keys()
    store.append(entry)
    save_keys(store)


def get_key_entry(key_id: str) -> dict:
    store = load_keys()
    for entry in store:
        if entry["key_id"] == key_id:
            return entry

    raise ValueError("Key ID not found")


In [7]:
import importlib
import KeyStorage

importlib.reload(KeyStorage)


<module 'KeyStorage' from '/Users/kirtikapuniani/Cryptography-Testing/KeyStorage.py'>

In [8]:
from KeyRotation import key_rotation
key_rotation()

'dek_20260131192937'

In [9]:
# KeyRotation

from datetime import datetime, timedelta
from cryptography.fernet import Fernet
from uuid import uuid4

from KeyStorage import load_keys, add_key_entry
from MasterKey import get_master_key

ROTATION_DAYS = 45


def _encrypt_dek_with_master(dek: bytes) -> str:
    master_fernet = Fernet(get_master_key())
    return master_fernet.encrypt(dek).decode()


def _generate_new_dek_entry() -> dict:
    dek = Fernet.generate_key()

    return {
        "key_id": str(uuid4()),
        "creation_date": datetime.now().isoformat(),
        "encrypted_key": _encrypt_dek_with_master(dek),
        "status": "active"
    }


def key_rotation() -> str:
    store = load_keys()
    now = datetime.now()

    if store:
        latest = max(store, key=lambda x: x["creation_date"])
        creation_date = datetime.fromisoformat(latest["creation_date"])

        if now - creation_date < timedelta(days=ROTATION_DAYS):
            return latest["key_id"]

        latest["status"] = "expired"

    new_entry = _generate_new_dek_entry()
    add_key_entry(new_entry)

    return new_entry["key_id"]


ImportError: cannot import name 'load_keys' from 'KeyStorage' (/Users/kirtikapuniani/Cryptography-Testing/KeyStorage.py)

In [ ]:
from KeyRotation import key_rotation

key_rotation()